In [1]:
%load_ext autoreload
%autoreload 2

from src.langgraph_agent import run_langgraph_agent


In [2]:

# Same test queries you used on the raw agent — behavior should be identical
# Test 1 -- single tool call
run_langgraph_agent("What is retrieval augmented generation?")


Tool sequence: ['search_book']

Final answer: **Retrieval Augmented Generation (RAG)** is a technique that enhances AI model responses by retrieving relevant information from external data sources before generating an answer.

## Key Concepts:

**The Problem It Solves:**
RAG addresses knowledge-intensive tasks where all available knowledge can't be directly input into a model. Without relevant context, AI models are more likely to make mistakes and "hallucinate" (generate false information).

**How It Works:**
The retrieve-then-generate pattern operates in two steps:
1. **Retrieve**: A retriever searches external data sources and returns only the information most relevant to the user's query
2. **Generate**: The model uses this retrieved information as context to generate a more accurate, detailed response

**Benefits:**
According to the research that coined the term (Lewis et al., 2020), RAG helps models:
- Generate more detailed and accurate responses
- Reduce hallucinations by provi

{'final_answer': '**Retrieval Augmented Generation (RAG)** is a technique that enhances AI model responses by retrieving relevant information from external data sources before generating an answer.\n\n## Key Concepts:\n\n**The Problem It Solves:**\nRAG addresses knowledge-intensive tasks where all available knowledge can\'t be directly input into a model. Without relevant context, AI models are more likely to make mistakes and "hallucinate" (generate false information).\n\n**How It Works:**\nThe retrieve-then-generate pattern operates in two steps:\n1. **Retrieve**: A retriever searches external data sources and returns only the information most relevant to the user\'s query\n2. **Generate**: The model uses this retrieved information as context to generate a more accurate, detailed response\n\n**Benefits:**\nAccording to the research that coined the term (Lewis et al., 2020), RAG helps models:\n- Generate more detailed and accurate responses\n- Reduce hallucinations by providing factua

In [3]:
# Test 2 -- multiple tool calls
run_langgraph_agent("Summarize what the book says about fine-tuning in 3 bullet points.")


Tool sequence: ['search_book', 'summarize_passage']

Final answer: Here's a 3-bullet summary of what the book says about fine-tuning:

• **Fine-tuning adapts models to specific needs; model merging complements it by combining multiple fine-tuned models for shared purposes.**

• **Full fine-tuning requires training all parameters but is memory-intensive; parameter-efficient techniques reduce memory footprint, increasing accessibility for practitioners.**

• **RAG outperforms fine-tuning for current information tasks; evaluation should precede expensive fine-tuning to ensure quality outcomes.**

These points are drawn from Chapter 7 (Finetuning) and related discussions in the book.


{'final_answer': "Here's a 3-bullet summary of what the book says about fine-tuning:\n\n• **Fine-tuning adapts models to specific needs; model merging complements it by combining multiple fine-tuned models for shared purposes.**\n\n• **Full fine-tuning requires training all parameters but is memory-intensive; parameter-efficient techniques reduce memory footprint, increasing accessibility for practitioners.**\n\n• **RAG outperforms fine-tuning for current information tasks; evaluation should precede expensive fine-tuning to ensure quality outcomes.**\n\nThese points are drawn from Chapter 7 (Finetuning) and related discussions in the book.",
 'tool_calls': [{'name': 'search_book',
   'input': {'query': 'fine-tuning', 'top_k': 5}},
  {'name': 'summarize_passage',
   'input': {'text': "Fine-tuning tailors one model to specific needs. While model merging is generally not considered finetuning, it's complementary to finetuning because it combines multiple models, often fine-tuned models, f

In [4]:
# Test 3 -- no tool calls
run_langgraph_agent("Hi, how are you?")

Tool sequence: []

Final answer: Hi! I'm doing well, thanks for asking! 😊

I'm here to help you learn about AI engineering. Whether you have questions about specific concepts, want to explore what the AI Engineering book covers, need summaries of key topics, or are curious about terminology—just let me know what you'd like to know!

What can I help you with today?


{'final_answer': "Hi! I'm doing well, thanks for asking! 😊\n\nI'm here to help you learn about AI engineering. Whether you have questions about specific concepts, want to explore what the AI Engineering book covers, need summaries of key topics, or are curious about terminology—just let me know what you'd like to know!\n\nWhat can I help you with today?",
 'tool_calls': []}

### Summary:

* Where does LangGraph save code compared to your raw agent?
 langGraph removes conversation-management boilerplate: the manual 
`for turn in range(max_turns)` loop, the message appending, the 
stop_reason checking, the tool_result formatting. My raw agent has 
about 40 lines dedicated to this; LangGraph absorbs it into the 
graph's execution engine. The net code count is similar because 
LangGraph adds state and edge definitions, but the *type* of code 
changed — from mechanical to structural.

* Where does it add code?
LangGraph adds three code categories:
1. state schema -- a TypedDict 'AgentState' declaring what data flows between nodes, plus a reducer (`Annotated[list, add]`) 
   telling LangGraph how to merge updates.
2. Node functions -- call tool, call model functions that takes state and returns a state-update dict. In the raw agent, these were inline in the loop.
3. Graph wiring. such as 'add node', 'add_edge', 'set_entry_point', 'add_conditional_edges'. This is the explicit control-flow that my raw agent had implicit in the loop structure.

* If you were building a fifth agent variant next week, which base would you extend — raw SDK or LangGraph? Why?
choose LangGraph for highly orchestrated, multi-agent, persistent, and observable workflows, especially where compliance, auditability, and visual workflow design and human-in-the-loop. The new agent has multiple decision points where the same tool could be called for different reasons. The new agent needs to route to different downstream models (cheap vs expensive). The new agent needs to persist state across sessions (LangGraph has built-in checkpointing). If I would like to change model mid-conversation.
choose raw SDK for developer-friendly, production-ready, and safety-focused agent systems, or where rapid prototyping, strong memory manangement, and advanced prompt handling are critical. Simple single-tool agents where the loop is 20 lines. One-off experiments where framework overhead slows iteration. 
Rule of thumb: use raw SDK until the loop grows if-statements, then migrate to LangGraph.